# --- Day 8: Haunted Wasteland ---
https://adventofcode.com/2023/day/8

You're still riding a camel across Desert Island when you spot a sandstorm quickly approaching. When you turn to warn the Elf, she disappears before your eyes! To be fair, she had just finished warning you about ghosts a few minutes ago.

One of the camel's pouches is labeled "maps" - sure enough, it's full of documents (your puzzle input) about how to navigate the desert. At least, you're pretty sure that's what they are; one of the documents contains a list of left/right instructions, and the rest of the documents seem to describe some kind of network of labeled nodes.

It seems like you're meant to use the left/right instructions to navigate the network. Perhaps if you have the camel follow the same instructions, you can escape the haunted wasteland!

After examining the maps for a bit, two nodes stick out: `AAA` and `ZZZ`. You feel like `AAA` is where you are now, and you have to follow the left/right instructions until you reach `ZZZ`.

This format defines each node of the network individually. For example:
```
RL

AAA = (BBB, CCC)
BBB = (DDD, EEE)
CCC = (ZZZ, GGG)
DDD = (DDD, DDD)
EEE = (EEE, EEE)
GGG = (GGG, GGG)
ZZZ = (ZZZ, ZZZ)
```

Starting with `AAA`, you need to look up the next element based on the next left/right instruction in your input. In this example, start with `AAA` and go right (R) by choosing the right element of `AAA`, `CCC`. Then, L means to choose the left element of `CCC`, `ZZZ`. By following the left/right instructions, you reach `ZZZ` in 2 steps.

Of course, you might not find `ZZZ` right away. If you run out of left/right instructions, repeat the whole sequence of instructions as necessary: `RL` really means `RLRLRLRLRLRLRLRL`... and so on. For example, here is a situation that takes 6 steps to reach `ZZZ`:
```
LLR

AAA = (BBB, BBB)
BBB = (AAA, ZZZ)
ZZZ = (ZZZ, ZZZ)
```

**Starting at `AAA`, follow the left/right instructions. How many steps are required to reach `ZZZ`?**

In [35]:
import re

# init list of directions 
directions = []
# init dict list of locations
locations = dict()
# init start and goal
start = 'AAA'
goal = 'ZZZ'

with open("input.txt", 'r') as f:
# with open("sample.txt", 'r') as f:
    lines = f.read().splitlines()
    # grab the directions
    directions= [char for char in lines.pop(0)]
    for line in lines[1:]:
        # regex match all the data we need
        items = re.match(r"^([A-Z]{3}) = \(([A-Z]{3}),\s([A-Z]{3})\)",line)
        # store them in a dict
        locations[items.group(1)] = [items.group(2), items.group(3)]

# convert all 'L'/'R' directions to 0/1
num_directions = []
for direction in directions:
    if direction == 'L':
        num_directions.append(0)
    else:
        num_directions.append(1)

# we start at the star
next = start
# found status
found = False
# nr of steps taken
step = 0

# while loop
while not found:
    # look for the next direction in the list, will be 0 or 1
    next_direction = num_directions[step%len(num_directions)]
    # next location lookup in dict
    next =locations[next][next_direction]
    # up the step counter
    step +=1
    # if goal is found break out of while loop
    if next == goal:
        found = True


# print the number of steps
print("The answer to part 1 is:",step, 'steps')

The answer to part 1 is: 20513 steps


# --- Part Two ---

The sandstorm is upon you and you aren't any closer to escaping the wasteland. You had the camel follow the instructions, but you've barely left your starting position. It's going to take significantly more steps to escape!

What if the map isn't for people - what if the map is for ghosts? Are ghosts even bound by the laws of spacetime? Only one way to find out.

After examining the maps a bit longer, your attention is drawn to a curious fact: the number of nodes with names ending in `A` is equal to the number ending in `Z`! If you were a ghost, you'd probably just start at every node that ends with `A` and follow all of the paths at the same time until they all simultaneously end up at nodes that end with Z.

For example:
```
LR

11A = (11B, XXX)
11B = (XXX, 11Z)
11Z = (11B, XXX)
22A = (22B, XXX)
22B = (22C, 22C)
22C = (22Z, 22Z)
22Z = (22B, 22B)
XXX = (XXX, XXX)
```
Here, there are two starting nodes, `11A` and `22A` (because they both end with `A`). As you follow each left/right instruction, use that instruction to simultaneously navigate away from both nodes you're currently on. Repeat this process until all of the nodes you're currently on end with `Z`. (If only some of the nodes you're on end with `Z`, they act like any other node and you continue as normal.) In this example, you would proceed as follows:
```
Step 0: You are at 11A and 22A.
Step 1: You choose all of the left paths, leading you to 11B and 22B.
Step 2: You choose all of the right paths, leading you to 11Z and 22C.
Step 3: You choose all of the left paths, leading you to 11B and 22Z.
Step 4: You choose all of the right paths, leading you to 11Z and 22B.
Step 5: You choose all of the left paths, leading you to 11B and 22C.
Step 6: You choose all of the right paths, leading you to 11Z and 22Z.
```

So, in this example, you end up entirely on nodes that end in Z after 6 steps.

**Simultaneously start on every node that ends with A. How many steps does it take before you're only on nodes that end with Z?**

In [156]:
import re
from collections import Counter
import math
import numpy as np

# init list of directions
directions = []
# init dict list of locations
locations = dict()

with open("input.txt", "r") as f:
    # with open("sample2.txt", 'r') as f:
    lines = f.read().splitlines()
    # grab the directions
    directions = [char for char in lines.pop(0)]
    for line in lines[1:]:
        # regex match all the data we need
        items = re.match(r"^([A-Z\d]{3}) = \(([A-Z\d]{3}),\s([A-Z\d]{3})\)", line)
        # store them in a dict
        locations[items.group(1)] = [items.group(2), items.group(3)]

# convert all 'L'/'R' directions to [0,1]
num_directions = []
for direction in directions:
    if direction == "L":
        num_directions.append(0)
    else:
        num_directions.append(1)

# figure out the starting locations for part 2
starting_locations = []
for loc in locations:
    if loc[2] == "A":
        starting_locations.append(loc)

# for all starting locations find the amount of cycles that it takes to find a Z
until_z = dict()

# loop through all the starting locations
for loc in starting_locations:
    # keep track of steps taken
    steps_taken = 0
    next = loc
    # are we done
    done = False
    # loop until ""..Z" location is found
    while not done:
        next_direction = num_directions[steps_taken % len(num_directions)]
        next = locations[next][next_direction]
        steps_taken += 1
        # if found, store the steps taken in the dict for the starting location and exit the loop
        if next[2] == "Z":
            until_z[loc] = steps_taken
            done = True

print('We now have a mapping for each how many steps it takes to get to a Z from their starting location')
print(until_z)
print('\nWe can now calculate the lowest common multiplier between these numbers with some math (default python funcs)')
result = math.lcm(*list(until_z.values()))
print("The answer to part 2 is:",result, 'steps')

We now have a mapping for each how many steps it takes to get to a Z from their starting location
{'MSA': 14893, 'AAA': 20513, 'PKA': 22199, 'NBA': 19951, 'RHA': 17141, 'CDA': 12083}

We can now calculat the lowest common multiplier between these numbers with some math
The answer to part 2 is: 15995167053923 steps
